In [ ]:
import os
try:
  from google.colab import drive
  drive.mount('/content/drive')
  os.chdir("/content/drive/MyDrive/COMP530-Project")
except ImportError:
  pass

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

In [ ]:
PARSE_PER_ITER = 500000
FIELDS_TO_PARSE = [
    'wlan.fc.type',
    'wlan.fc.subtype',
    'wlan_radio.phy',
    'wlan_radio.data_rate',
    'wlan_radio.duration',
    'radiotap.length',
    'radiotap.datarate',
    'wlan_radio.signal_dbm',
    'frame.time_relative',
    'frame.time_epoch',
    'frame.len',
    'radiotap.timestamp.ts',
    'wlan_radio.start_tsf',
    'radiotap.mactime',
    'wlan_radio.end_tsf',
    'wlan_radio.timestamp',
    'wlan_radio.channel',
    'wlan_radio.frequency',
    # Flags and string data
    'wlan.fc.ds',
    'wlan.fc.protected',
    'wlan.fc.moredata',
    'wlan.fc.frag',
    'wlan.fc.retry',
    'wlan.fc.pwrmgt',
    'radiotap.channel.flags.ofdm',
    'radiotap.channel.flags.cck',
]
FIELDS_TO_PARSE_BIN = ['wlan.fc.ds']

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import re

def parse_csv(file_path, label_condition, attack, normal="normal"):
  parsed = 0
  output_df = pd.DataFrame()
  scaler = MinMaxScaler()
  while True:
    df = pd.read_csv(
      file_path,
      escapechar='\\',
      low_memory=False,
      skiprows=range(1, parsed),
      nrows=PARSE_PER_ITER
    )

    if df.empty:
        break

    # Add label based on dynamic condition
    # Convert the label_condition string to a pandas query string format
    pandas_query_condition = label_condition.replace('||', '|').replace("&&", "&")
    pandas_query_condition = re.sub(r'\b(\w+\.\w+(?:\.\w+)?)\b', r'`\1`', pandas_query_condition)
    # Initialize 'label' column with 0 (negative case)
    df['label'] = normal
    # Set 'label' to 1 where the condition is met using .query() and .loc
    try:
        df.loc[df.query(pandas_query_condition).index, 'label'] = attack
    except Exception as e:
        print(f"Error applying label condition '{label_condition}': {e}")
        # Optionally, you might want to raise the error or handle it differently
        # For now, it will proceed with all labels as 0 if condition fails


    df_selected = df[FIELDS_TO_PARSE + ['label']].copy()

    # Parse boolean to 0,1 (explicitly convert boolean columns to int)
    for col in df_selected.columns:
        if df_selected[col].dtype == bool:
            df_selected[col] = df_selected[col].astype(int)

    # Parse binary fields (e.g., 'wlan.fc.ds' from hex to int)
    for i in FIELDS_TO_PARSE_BIN:
      if i in df_selected.columns:
        # Convert to string first to handle potential non-string values or NaNs,
        # then apply int(base=16) with error handling.
        df_selected[i] = df_selected[i].astype(str).apply(
            lambda x: int(x, 16) if isinstance(x, str) and x.strip() else 0 # Default to 0 for invalid/empty hex
        )


    output_df = pd.concat([output_df, df_selected], ignore_index=True)

    if df_selected.shape[0] < PARSE_PER_ITER:
        parsed += df_selected.shape[0]
        print(f"Parsed {parsed} rows")
        break

    parsed += PARSE_PER_ITER

    print(f"Parsed {parsed} rows")

  # Apply scaling after all data is parsed for consistent scaling across the entire dataset
  if not output_df.empty and FIELDS_TO_PARSE:
      # Ensure target columns are numeric before scaling
      existing_fields_to_parse = [field for field in FIELDS_TO_PARSE if field in output_df.columns]
      for col in existing_fields_to_parse:
          # Coerce non-numeric values to NaN, which MinMaxScaler can handle (e.g., by skipping or raising error depending on version)
          # A more robust approach might be to fill NaNs here if they are expected.
          output_df[col] = pd.to_numeric(output_df[col], errors='coerce')

      # Drop rows with NaNs in scaling columns, or fill them, before fit_transform if necessary
      # For simplicity, assuming MinMaxScaler can handle NaNs or data is clean.
      # If `fit_transform` raises an error due to NaNs, further NaN handling will be needed.
      if existing_fields_to_parse:
          output_df[existing_fields_to_parse] = scaler.fit_transform(output_df[existing_fields_to_parse])


  return output_df, scaler

In [ ]:
filename = input("Enter the filename: ") or "Deauth"
label_condition = input("Enter the label condition:") or "(wlan.fc.subtype==10 || wlan.fc.subtype==12) && wlan.fc.protected==0 && (frame.number >=1088022 && frame.number <=1626254)"
attack_label = input("Enter the attack label:") or "deauth"

In [ ]:
df, scaler = parse_csv(
    f"AWID3_Parsed/{filename}.csv",
    label_condition or "(wlan.fc.subtype==10 || wlan.fc.subtype==12) && wlan.fc.protected==0 && (frame.number >=1088022 && frame.number <=1626254)",
    attack_label,
    )

In [ ]:
df.describe().T

In [ ]:
df['label'].value_counts()

In [ ]:
import os
import pickle

output_dir = "AWID3_Processed"
os.makedirs(output_dir, exist_ok=True)

# Save the DataFrame to a CSV file
df.to_csv(os.path.join(output_dir, f"{filename}.csv"), index=False)
print(f"DataFrame saved to {os.path.join(output_dir, f'{filename}.csv')}")

# Save the scaler object
with open(os.path.join(output_dir, f"{filename}.pkl"), 'wb') as f:
    pickle.dump(scaler, f)
print(f"Scaler saved to {os.path.join(output_dir, f'{filename}.pkl')}")